In [3]:
import logging
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

# 1. Logging Setup Configuration
logging.basicConfig(
    filename="production_pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

app = FastAPI(title="ML Deployment Pipeline - Week 8")

# Baseline / Training Data Distribution (Example stats stored during training)
TRAIN_FEATURE_DISTRIBUTION = np.random.normal(loc=0.0, scale=1.0, size=1000)

class RequestPayload(BaseModel):
    features: list[float]

# Helper function: Data Drift Detection via KS-Test
def monitor_data_drift(incoming_features, train_features, threshold=0.05):
    stat, p_value = ks_2samp(incoming_features, train_features)
    if p_value < threshold:
        logging.warning(f"[MONITORING ALERT] Data Drift Detected! p-value: {p_value:.4f}")
        return True, p_value
    logging.info(f"[MONITORING NORMAL] Distribution matches training baseline. p-value: {p_value:.4f}")
    return False, p_value

@app.post("/predict")
def run_pipeline(payload: RequestPayload):
    # Stage 1: Raw Input Received
    logging.info("Stage 1: Raw Input Received")
    raw_data = payload.features

    if not raw_data:
        logging.error("Stage 1 Error: Empty input payload received.")
        raise HTTPException(status_code=400, detail="Empty feature list.")

    # Stage 2: Preprocessing
    try:
        logging.info("Stage 2: Preprocessing Started")
        processed_data = np.array(raw_data).reshape(1, -1)

        # Simple feature scaling / transformation logic
        scaled_features = (processed_data - np.mean(TRAIN_FEATURE_DISTRIBUTION)) / np.std(TRAIN_FEATURE_DISTRIBUTION)
        logging.info("Stage 2: Preprocessing Completed Successfully")
    except Exception as e:
        logging.error(f"Stage 2 Error: Preprocessing failed - {str(e)}")
        raise HTTPException(status_code=400, detail="Preprocessing Error")

    # Stage 3: Inference & Confidence Logging
    try:
        logging.info("Stage 3: Inference Started")

        # Simulation of model prediction and confidence score
        dummy_score = float(np.mean(scaled_features))
        prediction = 1 if dummy_score > 0.0 else 0
        confidence = float(min(1.0, abs(dummy_score) + 0.5))

        logging.info(f"Stage 3 Completed | Prediction: {prediction} | Confidence: {confidence:.4f}")
    except Exception as e:
        logging.error(f"Stage 3 Error: Model inference failed - {str(e)}")
        raise HTTPException(status_code=500, detail="Inference Error")

    # Stage 4: Basic Data Drift Monitoring
    drift_detected, p_val = monitor_data_drift(raw_data, TRAIN_FEATURE_DISTRIBUTION)

    # Stage 5: Response Generation
    logging.info("Stage 4: Pipeline Execution Complete. Returning Response.")
    return {
        "status": "success",
        "prediction": prediction,
        "confidence": confidence,
        "monitoring": {
            "drift_detected": drift_detected,
            "p_value": round(p_val, 4)
        }
    }

In [4]:
import logging
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

# Logging setup (Logs Google Colab ke file system mein pipeline.log name se save honge)
logging.basicConfig(
    filename="pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

print("Logging setup successfully configured!")

Logging setup successfully configured!


In [5]:
# Baseline Training Distribution Simulation
np.random.seed(42)
TRAIN_DISTRIBUTION = np.random.normal(loc=0.0, scale=1.0, size=1000)

def check_data_drift(live_batch, train_data, threshold=0.05):
    stat, p_value = ks_2samp(live_batch, train_data)
    if p_value < threshold:
        logging.warning(f"ALERT: Data Drift Detected! p-value: {p_value:.4f}")
        return True, p_value
    logging.info(f"Data Distribution Normal. p-value: {p_value:.4f}")
    return False, p_value

In [6]:
def run_end_to_end_pipeline(raw_input_data):
    # Stage 1: Raw Input Logging
    logging.info("Stage 1: Raw Input Received")
    print("Stage 1: Raw Input Received")

    # Stage 2: Preprocessing
    try:
        logging.info("Stage 2: Preprocessing Started")
        data_array = np.array(raw_input_data)
        # Standard Scaling Logic
        scaled_data = (data_array - np.mean(TRAIN_DISTRIBUTION)) / np.std(TRAIN_DISTRIBUTION)
        logging.info("Stage 2: Preprocessing Completed Successfully")
        print("Stage 2: Preprocessing Completed")
    except Exception as e:
        logging.error(f"Stage 2 Error: Preprocessing Failed - {str(e)}")
        return {"status": "error", "message": str(e)}

    # Stage 3: Model Inference & Confidence Logging
    try:
        logging.info("Stage 3: Inference Started")
        score = np.mean(scaled_data)
        prediction = int(score > 0.0)
        confidence = float(min(1.0, abs(score) + 0.5))

        logging.info(f"Stage 3 Completed | Prediction: {prediction} | Confidence: {confidence:.4f}")
        print(f"Stage 3 Completed: Prediction={prediction}, Confidence={confidence:.4f}")
    except Exception as e:
        logging.error(f"Stage 3 Error: Inference Failed - {str(e)}")
        return {"status": "error", "message": str(e)}

    # Stage 4: Basic Data Drift Check
    drift_flag, p_val = check_data_drift(data_array, TRAIN_DISTRIBUTION)

    # Stage 5: Response Logging
    logging.info("Stage 4: Pipeline Response Prepared Successfully")

    return {
        "prediction": prediction,
        "confidence": confidence,
        "drift_detected": drift_flag,
        "p_value": round(p_val, 4)
    }

# Test Run Normal Data
test_data_normal = np.random.normal(loc=0.0, scale=1.0, size=50)
response_1 = run_end_to_end_pipeline(test_data_normal)

# Test Run Shifted Data (Triggers Data Drift Alert)
test_data_drifted = np.random.normal(loc=2.5, scale=1.0, size=50)
response_2 = run_end_to_end_pipeline(test_data_drifted)

Stage 1: Raw Input Received
Stage 2: Preprocessing Completed
Stage 3 Completed: Prediction=1, Confidence=0.8145
Stage 1: Raw Input Received
Stage 2: Preprocessing Completed
Stage 3 Completed: Prediction=1, Confidence=1.0000


In [7]:
# Generated logs ko Colab output mein print karne ke liye
with open("pipeline.log", "r") as f:
    print(f.read())

2026-10-07 09:03:16,610 - INFO - Stage 1: Raw Input Received
2026-10-07 09:03:16,610 - INFO - Stage 2: Preprocessing Started
2026-10-07 09:03:16,611 - INFO - Stage 2: Preprocessing Completed Successfully
2026-10-07 09:03:16,611 - INFO - Stage 3: Inference Started
2026-10-07 09:03:16,611 - INFO - Stage 3 Completed | Prediction: 1 | Confidence: 0.8145
2026-10-07 09:03:16,617 - INFO - Data Distribution Normal. p-value: 0.0786
2026-10-07 09:03:16,617 - INFO - Stage 4: Pipeline Response Prepared Successfully
2026-10-07 09:03:16,618 - INFO - Stage 1: Raw Input Received
2026-10-07 09:03:16,618 - INFO - Stage 2: Preprocessing Started
2026-10-07 09:03:16,618 - INFO - Stage 2: Preprocessing Completed Successfully
2026-10-07 09:03:16,618 - INFO - Stage 3: Inference Started
2026-10-07 09:03:16,618 - INFO - Stage 3 Completed | Prediction: 1 | Confidence: 1.0000
2026-10-07 09:03:16,619 - WARNING - ALERT: Data Drift Detected! p-value: 0.0000
2026-10-07 09:03:16,619 - INFO - Stage 4: Pipeline Response

In [8]:
incident_content = """# Incident Response Plan: Silent Model Performance Degradation

## 1. Detection Strategy
- **Confidence Tracking:** Continuous logging and monitoring of prediction confidence scores[cite: 1].
- **Data Drift Detection:** Kolmogorov-Smirnov (KS-test) alerts triggered when live input distribution strays from training baseline ($p < 0.05$)[cite: 1].
- **Ground Truth Evaluation:** Periodic alignment of historical predictions with updated ground truth data.

## 2. Immediate Action Plan
- **Automated Alerts:** Send high-priority notifications to MLOps engineers upon drift detection.
- **Model Fallback:** Redirect prediction traffic to a stable previous checkpoint or baseline rule engine.

## 3. Recovery & Retraining Plan
- **Root Cause Analysis:** Inspect incoming feature schemas to isolate systemic pipeline failures vs natural drift.
- **Retraining Pipeline:** Execute automated retraining pipeline with freshly collected production data and redeploy after validation.
"""

with open("incident_response.md", "w") as f:
    f.write(incident_content)

print("incident_response.md created successfully!")

incident_response.md created successfully!


In [9]:
# GitHub details configured
!git config --global user.name "Your Name"
!git config --global user.email "your_email@gmail.com"

# Clone/Init repository
!git init
!git add .
!git commit -m "Add Week 8 end-to-end pipeline, drift monitoring, and incident response plan"
!git branch -M main
!git remote add origin https://<YOUR_GITHUB_TOKEN>@github.com/YOUR_USERNAME/Muhammad_Sahab_Week8.git
!git push -u origin main --force

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
[master (root-commit) 18e2d95] Add Week 8 end-to-end pipeline, drift monitoring, and incident response plan
 23 files changed, 51118 insertions(+)
 create mode 100644 .config/.last_opt_in_prompt.yaml
 create mode 100644 .config/.last_survey_prompt.yaml
 create mode 100644 .config/.last_update_check.json
 create mode 100644 .config/active_config
 create mode 100644 .config/config_sentinel
 create mode 100644 .config/configurations/config_default
 create mode 10